# Recipe + Shopping List Demo: A Guarded, Deterministic-Evaluator GER Agent

This notebook drives `adk.demos.recipe_shopping_list_demo.RecipeShoppingListAgent`, a concrete
`GenerateEvaluateReflectBase` subclass (`adk.generate_evaluate_reflect.base`) built on the same
**generate-evaluate-reflect** (GER) closed-loop pattern as
[`generate_eval_cases_demo.ipynb`](generate_eval_cases_demo.ipynb). Given the name of a dish, it:

1. **Refuses** anything that isn't "a recipe (and shopping list) for one named dish" - before
   the GER loop ever runs.
2. **Researches** 3-5 existing recipes for the dish via live Tavily search.
3. **Generates** a new recipe by comparing and contrasting that research, plus a shopping list
   for it.
4. **Evaluates** the shopping list against the recipe deterministically - no LLM judge, just a
   set comparison - and **reflects** any mismatch back into the next attempt.

The agent's own code lives in `src/adk/demos/recipe_shopping_list_demo.py` - this notebook
imports it and walks through what it does, rather than redefining any of the logic here.

Two things make this demo different from `generate_eval_cases_demo.ipynb`, the repo's other
concrete GER agent:

- **A guardrail ahead of the loop.** `GenerateEvaluateReflectBase`'s graph topology has no hook
  for a pre-generate check, so the "is this even a recipe request?" gate
  (`build_request_classifier`) runs in `RecipeShoppingListAgent.handle_request` *before*
  `.invoke()` is ever called - an out-of-scope request never touches the GER graph at all.
- **A deterministic evaluator.** "Does the shopping list match the recipe, with no duplicates?"
  has a mechanically checkable right answer, so `_ShoppingListConsistencyEvaluator` is a plain
  Python `Runnable` with no model call in it - proof that an `evaluate` node in this pattern can
  be *any* `Runnable` returning `eval_verdict`/`eval_rationale`, not necessarily another LLM
  call.

## Setup

Requires two API keys:

- `ANTHROPIC_API_KEY` — https://console.anthropic.com/
- `TAVILY_API_KEY` — https://tavily.com/

Copy `.env.example` (repo root) to `.env` and paste your keys in there — `load_dotenv()`
below loads it into this process. `.env` is gitignored, so real keys never get committed.
`adk.anthropic_client.get_anthropic_client()` and `adk.search_client.get_search_client()`
raise a clear `RuntimeError` if either key is still missing.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

for key in ("ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'MISSING'}")

ANTHROPIC_API_KEY: set
TAVILY_API_KEY: set


## The guardrail

`build_request_classifier` builds an `AnthropicRunnable` forced to call the `classify_request`
tool, so its output always parses into `{in_scope, dish, refusal_message}`. It runs once per
request, ahead of anything else.

In [2]:
from adk.demos.recipe_shopping_list_demo import (
    CLASSIFIER_SYSTEM_PROMPT,
    GENERATOR_SYSTEM_PROMPT,
    RecipeShoppingListAgent,
    search_existing_recipes,
)

print(CLASSIFIER_SYSTEM_PROMPT)

You gate requests for a recipe-and-shopping-list agent. The agent's ONLY capability is: given
the name of a dish, research existing recipes for it, synthesize one new recipe, and build a
shopping list for that recipe.

Given the user's message (as JSON, under "message"), decide whether it is asking for exactly
that - a recipe (optionally also a shopping list) for ONE named dish. If so, set "in_scope" to
true, extract the dish name into "dish", and leave "refusal_message" null. Otherwise - a
different kind of request, no dish named, multiple unrelated dishes, or anything outside recipe
development - set "in_scope" to false, leave "dish" null, and set "refusal_message" to a short,
polite statement of what this agent can do, so the user knows how to rephrase.

Submit your decision via the classify_request tool.



## Building the agent

Constructing `RecipeShoppingListAgent` builds and compiles the GER graph immediately
(`GenerateEvaluateReflectBase.__init__` calls `_build_generator`/`_build_evaluator`/
`_build_reflector`), so this cell needs a valid `ANTHROPIC_API_KEY`. The compiled topology is
the same closed-loop retry graph `generate_eval_cases_demo.ipynb` prints - what differs is what
runs *before* it, in `handle_request` below.

In [3]:
agent = RecipeShoppingListAgent()
agent.get_compiled_graph().get_graph().print_ascii()

          +-----------+          
          | __start__ |          
          +-----------+          
                *                
                *                
                *                
          +----------+           
          | generate |           
          +----------+           
           *         **          
         **            *         
        *               **       
+----------+              *      
| evaluate |              *      
+----------+...           *      
      .        ...        *      
      .           ....    *      
      .               ..  *      
  +------+          +---------+  
  | sink |          | reflect |  
  +------+          +---------+  
      *                          
      *                          
      *                          
+---------+                      
| __end__ |                      
+---------+                      


## Out of scope: the guardrail refuses

`handle_request` classifies the message first. Nothing dish-related here, so the classifier
rejects it and the GER graph never runs - no generator call, no search, no evaluator call.

In [4]:
refused = agent.handle_request("write me a haiku about spring")
refused

{'refused': True,
 'message': "I'm only able to help with creating a recipe and shopping list for a specific dish. I can't write poetry. Feel free to ask me for a recipe instead!"}

## In scope: research, then generate + evaluate + reflect

Now a real request. `handle_request` classifies it as in-scope, extracts the dish name, calls
`search_existing_recipes` (live Tavily search) once, then runs the GER loop with those results
as context for every attempt.

In [5]:
dish_preview = search_existing_recipes("chicken tikka masala")
for r in dish_preview:
    print(f"- {r['title']} ({r['url']})")

- Easy Chicken Tikka Masala | Olive & Mango (https://www.oliveandmango.com/easy-chicken-tikka-masala)
- Flavorful Chicken Tikka Masala Recipe - Souper Cubes (https://www.soupercubes.com/blogs/recipes/flavorful-chicken-tikka-masala)
- The Best Chicken Tikka Masala Recipe Out There - Cafe Delites (https://cafedelites.com/chicken-tikka-masala)
- Chicken Tikka Masala That Beats Any Takeout! (https://www.youtube.com/watch?v=r6C3JOaw6E0)
- My Restaurant Trick for Making “Perfect” Chicken Tikka Masala (It Blows Everyone Away!) (https://www.thekitchn.com/chicken-tikka-masala-recipe-23686912)


### The generator prompt

Given `context.dish`, `context.search_results` (the 3-5 recipes above), and - on a retry -
`reflection_feedback`, the generator compares and contrasts the research into one new recipe
plus a shopping list, via a forced `propose_recipe_and_shopping_list` tool call.

In [6]:
print(GENERATOR_SYSTEM_PROMPT)

You are a recipe developer. You will be given, as JSON: "context.dish" (the dish to develop),
"context.search_results" (3-5 existing recipes for that dish, each a {title, url, snippet}
scraped from a real recipe site), and possibly "reflection_feedback" (a critique of your
previous attempt, if any).

Compare and contrast the search results - where they agree, where they diverge on ingredients,
quantities, or technique - and synthesize ONE new recipe that reflects the best of what you
found, not a copy of any single source. Then build its shopping list: one line per ingredient
the recipe uses, with a matching name, quantity, and unit. The shopping list must cover every
ingredient the recipe calls for, contain nothing the recipe doesn't use, and list each
ingredient exactly once (combine an ingredient used in multiple steps into a single line rather
than repeating it).

If "reflection_feedback" is present, it names a specific mismatch between the recipe and the
shopping list from your pr

### Running the full request

This is the same `search_existing_recipes` call as above, followed by one `.invoke()` through
the compiled GER graph - `handle_request` just wires the two together and adds the guardrail.

In [7]:
request = "Find me a recipe for chicken tikka masala and put together a shopping list."
out = agent.handle_request(request)

print(f"Request: {request}")
print(f"Refused: {out['refused']}")
print(f"stop_reason: {out['stop_reason']}")
print(f"attempts made: {out['attempt_index']}")
print(f"final eval_verdict: {out['eval_verdict']}")
print(f"final eval_rationale: {out['eval_rationale']}")

Request: Find me a recipe for chicken tikka masala and put together a shopping list.
Refused: False
stop_reason: all_pass
attempts made: 2
final eval_verdict: pass
final eval_rationale: Shopping list covers every recipe ingredient exactly once, with no extras.


`eval_rationale` above comes straight out of `_ShoppingListConsistencyEvaluator` - a plain
set comparison between the recipe's ingredient names and the shopping list's item names, no LLM
call involved. Had it failed, that same rationale would have been reused verbatim as
`reflection_feedback` for the next attempt (`_RationaleReflector` - the same reuse-the-rationale
move `generate_eval_cases_demo._RationaleReflector` makes for its own evaluator).

### The accepted recipe and shopping list

`accepted_artifact` is the `{recipe, shopping_list}` candidate that passed evaluation.

In [8]:
artifact = out["accepted_artifact"]
recipe = artifact["recipe"]

print(f"Recipe: {recipe['name']}")
if recipe.get("servings"):
    print(f"Servings: {recipe['servings']}")

print("\nIngredients:")
for ingredient in recipe["ingredients"]:
    unit = f" {ingredient['unit']}" if ingredient.get("unit") else ""
    print(f"  - {ingredient['quantity']}{unit} {ingredient['name']}")

print("\nInstructions:")
for i, step in enumerate(recipe["instructions"], start=1):
    print(f"  {i}. {step}")

Recipe: Chicken Tikka Masala
Servings: 4-6

Ingredients:
  - 2 lbs boneless, skinless chicken thighs
  - 1 cup plain whole-milk yogurt
  - 6 cloves garlic, minced
  - 4 teaspoons fresh ginger, minced
  - 2 teaspoons ground turmeric
  - 2 teaspoons ground cumin
  - 2 teaspoons ground coriander
  - 3 teaspoons garam masala
  - 1 teaspoon cayenne or chili powder
  - 2 teaspoons salt
  - 2 tablespoons vegetable oil
  - 4 tablespoons unsalted butter
  - 1 large yellow onion, finely chopped
  - 1 28-ounce can crushed tomatoes
  - 0.5 cup water
  - 0.75 cup heavy cream
  - 1 teaspoon sugar
  - 1 tablespoon kasuri methi (dried fenugreek leaves), crushed
  - 0.25 cup fresh cilantro, chopped
  - 1 batch basmati rice or naan, for serving

Instructions:
  1. In a large bowl, whisk together yogurt, half the garlic (3 cloves), half the ginger (2 teaspoons), 1 teaspoon turmeric, 1 teaspoon cumin, 1 teaspoon coriander, 1 teaspoon garam masala, and 1 teaspoon salt.
  2. Cut chicken into 1.5-inch cubes,

In [9]:
print("Shopping list:")
for item in artifact["shopping_list"]:
    unit = f" {item['unit']}" if item.get("unit") else ""
    print(f"  - {item['quantity']}{unit} {item['name']}")

Shopping list:
  - 2 lbs boneless, skinless chicken thighs
  - 1 cup plain whole-milk yogurt
  - 6 cloves garlic, minced
  - 4 teaspoons fresh ginger, minced
  - 2 teaspoons ground turmeric
  - 2 teaspoons ground cumin
  - 2 teaspoons ground coriander
  - 3 teaspoons garam masala
  - 1 teaspoon cayenne or chili powder
  - 2 teaspoons salt
  - 2 tablespoons vegetable oil
  - 4 tablespoons unsalted butter
  - 1 large yellow onion, finely chopped
  - 1 28-ounce can crushed tomatoes
  - 0.5 cup water
  - 0.75 cup heavy cream
  - 1 teaspoon sugar
  - 1 tablespoon kasuri methi (dried fenugreek leaves), crushed
  - 0.25 cup fresh cilantro, chopped
  - 1 batch basmati rice or naan, for serving


## Try your own dish

Swap in any dish name - the guardrail only cares that the request names exactly one.

In [10]:
custom_out = agent.handle_request("I need a recipe and shopping list for masala dosa.")

print(f"Refused: {custom_out['refused']}")
print(f"stop_reason: {custom_out['stop_reason']}\n")

custom_recipe = custom_out["accepted_artifact"]["recipe"]
print(f"Recipe: {custom_recipe['name']}\n")

print("Ingredients:")
for ingredient in custom_recipe["ingredients"]:
    unit = f" {ingredient['unit']}" if ingredient.get("unit") else ""
    print(f"  - {ingredient['quantity']}{unit} {ingredient['name']}")

print("\nShopping list:")
for item in custom_out["accepted_artifact"]["shopping_list"]:
    unit = f" {item['unit']}" if item.get("unit") else ""
    print(f"  - {item['quantity']}{unit} {item['name']}")

Refused: False
stop_reason: all_pass

Recipe: Crispy Masala Dosa with Spiced Potato Filling

Ingredients:
  - 1.5 cups sona masuri rice (or idli rice)
  - 0.5 cup parboiled rice
  - 0.5 cup urad dal (whole white lentils)
  - 2 tbsp chana dal
  - 1 tbsp toor dal
  - 0.25 cup poha (flattened rice)
  - 0.5 tsp methi seeds (fenugreek)
  - 1.5 tsp salt
  - as needed water
  - 3 tbsp oil
  - 2 tbsp butter
  - 4 medium (boiled) potatoes
  - 1 large, sliced onion
  - 1 tsp mustard seeds
  - 0.5 tsp cumin seeds
  - 2-3 chopped green chilies
  - 1 tbsp, minced ginger
  - 10-12 leaves curry leaves
  - 1-2 dried red chilies
  - 0.5 tsp turmeric powder
  - 2 tbsp, chopped fresh coriander leaves
  - 1 tsp lemon juice
  - 1 tsp chana dal (for tempering)
  - 1 tsp urad dal (for tempering)

Shopping list:
  - 1.5 cups sona masuri rice (or idli rice)
  - 0.5 cup parboiled rice
  - 0.5 cup urad dal (whole white lentils)
  - 2 tbsp chana dal
  - 1 tbsp toor dal
  - 0.25 cup poha (flattened rice)
  - 0.5 t

## All attempts, including any rejected ones

`attempts` accumulates one entry per generate/evaluate cycle, so a request that needed a
reflect → regenerate pass shows more than one candidate here. Both requests above actually
took two attempts: the deterministic evaluator caught a real mismatch on the first try (a
missing or duplicated shopping-list item), `_RationaleReflector` passed that rationale straight
through as `reflection_feedback`, and the second attempt fixed it and passed.

In [11]:
for attempt in custom_out["attempts"]:
    print(f"attempt {attempt['attempt_index']}: {attempt['candidate']['recipe']['name']}")

attempt 1: Crispy Masala Dosa with Spiced Potato Filling
attempt 2: Crispy Masala Dosa with Spiced Potato Filling
